# OCVWorkChain: DFT with JSON input

Launches the OCV workflow at the level of PBEsol-DFT from a single OPTIMADE-compatible JSON file (structure, spin settings, k-points, magnetic moments). Example inputs are provided in `json_inputs`.

Requires aiida-open_circuit_voltage >= 0.7 with aiida-quantumespresso 5.x

Note: DFT+U+V (Hubbard) mode is not available through the JSON builder; use the Hubbard submission notebook for that.

## Loading libraries

In [ ]:
from aiida import load_profile, orm
## Indicate your profile name here
your_profile_name = 'develop'
load_profile(your_profile_name)
from aiida.plugins import WorkflowFactory
from aiida.engine import submit

## JSON input

In [ ]:
import json

time, num_machines, num_mpiprocs_per_machine, num_cores_per_mpiproc, npool = 83200, 2, 128, 1, 8

overrides = {"ocv_relax":{
                "base_relax":{
                    "pseudo_family": "SSSP/1.3/PBEsol/precision",
                    "pw":{
                        "parallelization":{
                            "npool": npool},}}},
            "scf":{
                "pseudo_family": "SSSP/1.3/PBEsol/precision",
                "pw":{
                    "parallelization":{
                        "npool": int(npool/num_machines)},}},}

OCVWorkChain = WorkflowFactory('quantumespresso.ocv.ocvwc')

builder = OCVWorkChain.get_builder_from_json('json_inputs/Li_olivine.json', overrides=overrides)

pw_dict = builder.ocv_relax['base_relax']['pw']
pw_dict['parameters']['ELECTRONS']['electron_maxstep'] = 100
pw_dict['metadata']['options']['max_wallclock_seconds'] = time
pw_dict['metadata']['options']['resources']['num_machines'] = num_machines
pw_dict['metadata']['options']['resources']['num_mpiprocs_per_machine'] = num_mpiprocs_per_machine
pw_dict['metadata']['options']['resources']['num_cores_per_mpiproc'] = num_cores_per_mpiproc

builder.ocv_relax['max_meta_convergence_iterations'] = orm.Int(10)

# Keep a small value as it finishes in a few minutes even on a small cluster
builder.scf['pw']['metadata']['options']['max_wallclock_seconds'] = 1800
builder.scf['pw']['metadata']['options']['resources']['num_machines'] = 1
builder.scf['pw']['metadata']['options']['resources']['num_mpiprocs_per_machine'] = num_mpiprocs_per_machine
builder.scf['pw']['metadata']['options']['resources']['num_cores_per_mpiproc'] = num_cores_per_mpiproc
builder.scf['pw']['parameters']['ELECTRONS']['electron_maxstep'] = 100

builder.ocv_parameters['distance'] = 8.0
builder.ocv_parameters['SOC_vc_relax'] = False
builder.ocv_parameters['SOC_relax_all_supercells'] = False

# Submitting the builder to launch the WorkChain
ocvwc_node = submit(builder)
print(f'Submitted OCVWorkChain PK={ocvwc_node.pk}')

## JSON output

In [ ]:
## Once the workchain is finished, the output dictionary can then be dumped as json file
with open('result.json', 'w') as to_write:
    res_d = ocvwc_node.outputs['common_workflow_output'].get_dict()
    json.dump(res_d, to_write, indent=4, sort_keys=True)